# 농사로 실내정원용 식물 전체 받기

**준비**: 왼쪽 열쇠 아이콘(Secrets)에 `NONGSARO_API_KEY`를 넣고 노트북 액세스를 켜요.

약 200종이면 요청이 200번 정도이고 몇 분 걸려요. 우리 30종 중 실내식물 10종도 이 파일 안에 들어 있어서, 이 노트북 하나만 실행하면 돼요.


In [ ]:
# 농사로 실내정원용 식물 API: 전체 목록 + 상세 정보 받기
# 준비: 코랩 열쇠 아이콘(Secrets)에 NONGSARO_API_KEY 저장
import json, time, urllib.request
import xml.etree.ElementTree as ET
from google.colab import userdata

KEY = userdata.get("NONGSARO_API_KEY").strip()
BASE = "http://api.nongsaro.go.kr/service/garden"

def fetch(url):
    xml = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")
    root = ET.fromstring(xml)
    items = [{c.tag: (c.text or "").strip() for c in it} for it in root.iter("item")]
    return items, root

# 1) 전체 목록 (한 번에 100개씩)
plants, page = [], 1
while True:
    items, root = fetch(f"{BASE}/gardenList?apiKey={KEY}&numOfRows=100&pageNo={page}")
    if not items:
        if page == 1:
            print("목록이 비어 있어요:", root.findtext(".//resultMsg"))
        break
    plants += items
    total = root.findtext(".//totalCount")
    print(f"{page}쪽: {len(items)}개 (누적 {len(plants)} / 전체 {total})")
    if total and len(plants) >= int(total):
        break
    page += 1
    time.sleep(0.3)

# 2) 식물마다 상세 정보
details = []
for i, p in enumerate(plants, 1):
    d, _ = fetch(f"{BASE}/gardenDtl?apiKey={KEY}&cntntsNo={p.get('cntntsNo')}")
    row = {"cntntsNo": p.get("cntntsNo"), "cntntsSj": p.get("cntntsSj")}
    row.update(d[0] if d else {})
    details.append(row)
    if i % 20 == 0:
        print(f"상세 {i}/{len(plants)}")
    time.sleep(0.3)

text = json.dumps(details, ensure_ascii=False, indent=1).replace(KEY, "REDACTED")  # 키가 파일에 남지 않게
open("nongsaro_garden_all.json", "w", encoding="utf-8").write(text)
print(f"완료: {len(details)}종 → nongsaro_garden_all.json")


In [ ]:
from google.colab import files
files.download('nongsaro_garden_all.json')
